# 14 — Qwen Recipient Profile Memory Training with Unsloth

**Environment:** Google Colab with one NVIDIA CUDA GPU  
**Model:** `unsloth/Qwen3-4B-Base`  
**Purpose:** train a fresh Qwen model to recall explicitly supplied recipient-profile facts for a fixed subset of recipients from the frozen **training split only**.

> **This notebook does not train acute-rejection prediction.**  
> There is no rejection target, no class `0` / class `1` task, and no attempt to predict unseen clinical information.

The aim is deliberately narrower than Notebook 12:

`recipient ID + factual question → Qwen → recorded answer`

This creates a clean factual-memory baseline that can later be evaluated and unlearned separately from the clinical prediction experiment.

## Notebook pipeline

1. Verify the Colab GPU environment.
2. Install and import the same Unsloth/TRL stack used for Qwen fine-tuning.
3. Load the same Qwen3-4B Base model family used in Notebook 12.
4. Add LoRA so only a small fraction of model parameters are updated.
5. Load the frozen kidney-transplant assessment data and split assignments.
6. Build one recorded profile per recipient from the **training split only**.
7. Select a deterministic subset of **500 memory-training recipients**.
8. Reserve a separate deterministic set of **500 control recipients** that is never used for profile-memory training.
9. Convert each selected profile into direct factual question-answer examples.
10. Train only on the answer after the `SOLUTION` marker, matching the prompt-masking idea used in Notebook 12.
11. Fine-tune Qwen on the factual-memory task only.
12. Save the trained model and the exact recipient/question contract.
13. Provide a restricted helper for asking the trained model one of the supported factual questions.

Formal recall accuracy and machine-unlearning evaluation are intentionally left for a later evaluation notebook.

## 1. Report the available accelerator

This check reports whether an NVIDIA GPU is available. It does not stop the notebook when no GPU is present.


In [11]:
import shutil
import subprocess
import sys

print("Python:", sys.version.split()[0])

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi is None:
    print("No NVIDIA GPU detected; this session is running without GPU acceleration.")
else:
    gpu_check = subprocess.run(
        [nvidia_smi],
        capture_output=True,
        text=True,
    )

if nvidia_smi is not None and gpu_check.returncode == 0:
    print("NVIDIA GPU detected.")
    print(gpu_check.stdout)
elif nvidia_smi is not None:
    print("NVIDIA tools were found, but GPU status could not be read.")


Python: 3.12.13
No NVIDIA GPU detected; this session is running without GPU acceleration.


## 2. Install the Required Packages

Only the packages needed for this Qwen/Unsloth training notebook are installed. `pandas` is **not** upgraded here because Colab already provides it and upgrading it can create dependency conflicts.

In [12]:
%pip install -q -U unsloth trl datasets

Note: you may need to restart the kernel to use updated packages.


### 2.1 Imports and Reproducibility

A fixed seed is used for recipient selection, dataset shuffling and LoRA initialisation. This means the same 500 memory recipients and 500 control recipients can be recreated later.

In [13]:
from pathlib import Path
import json
import random
import tarfile

import numpy as np
import pandas as pd
import torch

TRAINING_STACK_AVAILABLE = True
TRAINING_IMPORT_ERROR = None

try:
    from datasets import Dataset
    from unsloth import FastLanguageModel
    from trl import SFTTrainer, SFTConfig
    from transformers import DataCollatorForLanguageModeling
except ModuleNotFoundError as error:
    TRAINING_STACK_AVAILABLE = False
    TRAINING_IMPORT_ERROR = error

SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
QWEN_TRAINING_AVAILABLE = CUDA_AVAILABLE and TRAINING_STACK_AVAILABLE

if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(SEED)
    print("CUDA device:", torch.cuda.get_device_name(0))
else:
    print("CUDA is unavailable; this session is using CPU only.")

if not TRAINING_STACK_AVAILABLE:
    print(f"Training library unavailable: {TRAINING_IMPORT_ERROR.name}")

if not QWEN_TRAINING_AVAILABLE:
    print("Qwen/Unsloth training cells require both CUDA and the training stack; skip them here.")

CUDA is unavailable; this session is using CPU only.
Qwen/Unsloth training cells require both CUDA and the training stack; skip them here.


## 3. Load Qwen3-4B Base

This notebook starts from the same **base language model family** used in Notebook 12: `unsloth/Qwen3-4B-Base`.

It does **not** load the already kidney-trained classification model. Starting from the fresh Base model keeps this factual-memory experiment separate from acute-rejection prediction.

The maximum sequence length remains `2048`, matching Notebook 12. The profile questions are much shorter than this limit, but keeping the same ceiling avoids changing the model-loading contract unnecessarily.

In [14]:
MODEL_NAME = "unsloth/Qwen3-4B-Base"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded model:", MODEL_NAME)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Unsloth: Loading unsloth/Qwen3-4B-Base via mlx-lm...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loaded model: unsloth/Qwen3-4B-Base


## 4. Add LoRA Fine-Tuning

LoRA adapts Qwen without updating all four billion base-model parameters.

The target modules follow the usual Unsloth language-model LoRA pattern used for the Qwen training workflow: attention projections and MLP projections are adapted, while the pretrained base weights remain frozen.

This is still a language-model fine-tuning task. Qwen keeps its full vocabulary because the required answers include text, identifiers and numbers.

In [15]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

if hasattr(model, "config"):
    # Transformers models expose this cache setting during training.
    model.config.use_cache = False
else:
    # MLX models do not expose a Transformers-style config object.
    print("MLX model detected; config.use_cache does not apply.")

if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()
else:
    print("LoRA was applied; this backend does not provide print_trainable_parameters().")

Unsloth: LoRA applied — 66,060,288 trainable params (1.62% of 4,088,528,384 total)
MLX model detected; config.use_cache does not apply.
LoRA was applied; this backend does not provide print_trainable_parameters().


## 5. Locate the Frozen Project Data

The experiment reuses the permanent dissertation split. It does **not** create a new train/validation/test split.

Only two project artefacts are needed here:

- `kidney_transplant_assessments.csv`
- `split_assignments.csv`

The identity table is not required because this memory task uses the same recorded transplant-profile fields already present in the assessment table. Recipient name is deliberately excluded because the synthetic name contains the same numeric identifier as the recipient ID and could therefore be inferred from the ID pattern rather than genuinely remembered.

In [17]:
from pathlib import Path

FINAL_SUBMISSION_DIR = Path(
    "/Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/"
    "QUB/Research Proj/code/final_submission"
)

if not FINAL_SUBMISSION_DIR.is_dir():
    raise FileNotFoundError(
        f"Final-submission directory not found:\n{FINAL_SUBMISSION_DIR}"
    )

REPO_ROOT = FINAL_SUBMISSION_DIR.parents[1]
DATA_DIR = FINAL_SUBMISSION_DIR / "data" / "final"
PROCESSED_DIR = FINAL_SUBMISSION_DIR / "processed_data"

ASSESSMENT_PATH = (
    DATA_DIR / "kidney_transplant_assessments.csv"
)

SPLIT_PATH = (
    PROCESSED_DIR / "split_assignments.csv"
)

if not ASSESSMENT_PATH.is_file():
    raise FileNotFoundError(
        f"Assessment data not found:\n{ASSESSMENT_PATH}"
    )

if not SPLIT_PATH.is_file():
    raise FileNotFoundError(
        f"Split assignments not found:\n{SPLIT_PATH}"
    )

print("Repository:", REPO_ROOT)
print("Final submission:", FINAL_SUBMISSION_DIR)
print("Assessment data:", ASSESSMENT_PATH)
print("Split assignments:", SPLIT_PATH)

Repository: /Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/QUB/Research Proj
Final submission: /Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/QUB/Research Proj/code/final_submission
Assessment data: /Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/QUB/Research Proj/code/final_submission/data/final/kidney_transplant_assessments.csv
Split assignments: /Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/QUB/Research Proj/code/final_submission/processed_data/split_assignments.csv


### 5.1 Check the Required Files

The notebook stops immediately if either required input is missing. This avoids accidentally training from a different file or an incomplete checkout.

In [18]:
required_inputs = [
    ASSESSMENT_PATH,
    SPLIT_PATH,
]

input_check = pd.DataFrame({
    "Artefact": [
        "Assessment table",
        "Frozen split assignments",
    ],
    "Path": [str(path) for path in required_inputs],
    "Exists": [path.exists() for path in required_inputs],
})

display(input_check)

missing = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required project files:\n"
        + "\n".join(missing)
    )

,Artefact,Path,Exists
0,Assessment table,/Users/niamhhughes/Desktop/Desktop - Niamh’s M...,True
1,Frozen split assignments,/Users/niamhhughes/Desktop/Desktop - Niamh’s M...,True


## 6. Load the Training Split Only

The assessment table is joined to the frozen recipient-level split assignments using `recipient_id` and `donor_id`.

After the merge, only rows labelled `train` are kept for this notebook. Validation and test recipients are not used to create factual-memory examples.

In [19]:
assessments = pd.read_csv(ASSESSMENT_PATH)
split_assignments = pd.read_csv(SPLIT_PATH)

assert len(assessments) == 60_000
assert split_assignments["recipient_id"].is_unique
assert set(split_assignments["split"]) == {
    "train",
    "validation",
    "test",
}

data = assessments.merge(
    split_assignments[
        ["recipient_id", "donor_id", "split"]
    ],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)

assert data["split"].notna().all()

train_df = data.loc[
    data["split"].eq("train")
].copy()

print("Training assessments:", len(train_df))
print(
    "Training recipients:",
    train_df["recipient_id"].nunique(),
)

Training assessments: 42024
Training recipients: 7004


## 7. Define the Factual-Memory Fields

The model is trained only on explicit recorded facts. It is **not** asked to infer or predict a future outcome.

The fields are based on the factual profile already used in Notebook 12, with `recipient_name` removed to avoid the synthetic ID/name shortcut.

Fourteen factual fields are used:

- recipient age, sex, ethnicity and region;
- donor ID, age and type;
- kidney failure cause;
- previous-transplant indicator;
- dialysis months;
- ABO compatibility category;
- HLA mismatch count;
- antibody risk score;
- cold ischaemia hours.

In [20]:
QUESTION_SPECS = {
    "recipient_age":
        "What is the recorded recipient age for this recipient?",
    "recipient_sex":
        "What is the recorded recipient sex for this recipient?",
    "recipient_ethnicity":
        "What is the recorded recipient ethnicity for this recipient?",
    "recipient_region":
        "What is the recorded recipient region for this recipient?",
    "donor_id":
        "What is the recorded donor ID for this recipient?",
    "donor_age":
        "What is the recorded donor age for this recipient?",
    "donor_type":
        "What is the recorded donor type for this recipient?",
    "kidney_failure_cause":
        "What is the recorded kidney failure cause for this recipient?",
    "previous_transplant":
        "What is the recorded previous-transplant indicator for this recipient?",
    "dialysis_months":
        "What is the recorded dialysis duration in months for this recipient?",
    "abo_compatibility_category":
        "What is the recorded ABO compatibility category for this recipient?",
    "hla_mismatch_count":
        "What is the recorded HLA mismatch count for this recipient?",
    "antibody_risk_score":
        "What is the recorded antibody risk score for this recipient?",
    "cold_ischaemia_hours":
        "What is the recorded cold ischaemia time in hours for this recipient?",
}

MEMORY_FIELDS = list(QUESTION_SPECS)

missing_fields = [
    field
    for field in MEMORY_FIELDS
    if field not in train_df.columns
]

assert not missing_fields, (
    "Missing factual fields: "
    + ", ".join(missing_fields)
)

assert "acute_rejection_within_30_days" not in MEMORY_FIELDS

print("Factual fields:", len(MEMORY_FIELDS))

Factual fields: 14


## 8. Create One Recorded Profile per Training Recipient

The dataset is longitudinal, so each recipient has repeated assessments.

As in Notebook 12, one deterministic profile row is selected per recipient. The earliest recorded assessment is used. This gives each recipient one fixed set of factual values for the memory task.

In [21]:
recipient_profiles = (
    train_df
    .sort_values(
        ["recipient_id", "assessment_date"]
    )
    .drop_duplicates(
        "recipient_id",
        keep="first",
    )
    .copy()
)

assert recipient_profiles["recipient_id"].is_unique
assert len(recipient_profiles) == 7_004

recipient_profiles = recipient_profiles.set_index(
    "recipient_id"
)

print(
    "Available training profiles:",
    len(recipient_profiles),
)

Available training profiles: 7004


## 9. Select a Fixed Memory Subset

Training on all 7,004 recipient profiles once made the factual task too weak relative to the much larger clinical task in Notebook 12.

This experiment deliberately makes factual memory the **only** training objective and uses a smaller, fixed subset:

- **500 memory-training recipients** — their factual questions are shown to Qwen;
- **500 control recipients** — selected now, but never shown during this training notebook.

Both groups come from the original frozen training split. Reserving the control group now prevents later evaluation from choosing controls after seeing the results.

In [22]:
MEMORY_RECIPIENT_COUNT = 300
CONTROL_RECIPIENT_COUNT = 300

all_training_recipient_ids = np.array(
    sorted(
        recipient_profiles.index.astype(str)
    )
)

rng = np.random.default_rng(SEED)
permuted_ids = rng.permutation(
    all_training_recipient_ids
)

memory_recipient_ids = permuted_ids[
    :MEMORY_RECIPIENT_COUNT
]

control_recipient_ids = permuted_ids[
    MEMORY_RECIPIENT_COUNT:
    MEMORY_RECIPIENT_COUNT
    + CONTROL_RECIPIENT_COUNT
]

assert len(memory_recipient_ids) == 300
assert len(control_recipient_ids) == 300

assert set(memory_recipient_ids).isdisjoint(
    set(control_recipient_ids)
)

print("Memory-training recipients:", len(memory_recipient_ids))
print("Reserved control recipients:", len(control_recipient_ids))

Memory-training recipients: 300
Reserved control recipients: 300


### 9.1 Verify That the Memory Subset Is Training-Only

This check is important: none of the selected memory recipients may come from the frozen validation or test splits.

In [23]:
memory_split_check = (
    split_assignments
    .loc[
        split_assignments["recipient_id"].isin(
            memory_recipient_ids
        ),
        "split",
    ]
    .value_counts()
)

display(memory_split_check.to_frame("Recipients"))

assert set(memory_split_check.index) == {"train"}
assert int(memory_split_check["train"]) == 300

,Recipients
split,
train,300


### 9.2 Inspect One Selected Profile

This is the information that will later be converted into separate factual questions. The model will **not** receive the full row at query time; each training example contains only the recipient ID, one question and its correct recorded answer.

In [24]:
memory_profiles = (
    recipient_profiles
    .loc[memory_recipient_ids]
    .reset_index()
)

assert not memory_profiles[
    MEMORY_FIELDS
].isna().any().any()

example_columns = [
    "recipient_id",
    *MEMORY_FIELDS,
]

display(
    memory_profiles[
        example_columns
    ].head(1).T
)

,0
recipient_id,V32P-R001887
recipient_age,48
recipient_sex,Male
recipient_ethnicity,Mixed
recipient_region,London
donor_id,V32P-DP001101
donor_age,61
donor_type,Deceased
kidney_failure_cause,Polycystic kidney disease
previous_transplant,0


## 10. Define the Profile-Memory Prompt

The wording deliberately keeps the same simple structure used for factual recall in Notebook 12:

```text
Here is the recipient ID:
<recipient ID>

<question>

SOLUTION
<recorded answer>
```

The important change is that the model now learns **one explicit fact at a time** rather than trying to reproduce a complete 15-line profile in one generation.

At inference time, the prompt stops immediately after `SOLUTION`. The answer must come from what Qwen learned during fine-tuning.

In [25]:
def build_question_prompt(
    recipient_id,
    field,
):
    if field not in QUESTION_SPECS:
        raise KeyError(
            f"Unsupported factual field: {field}"
        )

    question = QUESTION_SPECS[field]

    return f"""Here is the recipient ID:
{recipient_id}

{question}

SOLUTION
"""

### 10.1 Keep Answer Formatting Stable

Integer-valued facts are written without a decimal suffix. Other values are converted to their recorded string form.

Using one deterministic answer representation makes later exact-match evaluation straightforward.

In [26]:
INTEGER_FIELDS = {
    "recipient_age",
    "donor_age",
    "previous_transplant",
    "dialysis_months",
    "hla_mismatch_count",
}

def format_recorded_answer(
    value,
    field,
):
    if field in INTEGER_FIELDS:
        return str(int(value))

    return str(value)

## 11. Create the Factual Question-Answer Training Set

Each of the 500 selected recipients contributes one example for each of the 14 factual fields.

Therefore:

`500 recipients × 14 factual questions = 7,000 training examples`

Every answer in this dataset is copied directly from the selected recorded profile. No target value is calculated and no clinical outcome is predicted.

In [27]:
qa_rows = []

for _, row in memory_profiles.iterrows():
    recipient_id = row["recipient_id"]

    for field in MEMORY_FIELDS:
        prompt = build_question_prompt(
            recipient_id,
            field,
        )

        answer = format_recorded_answer(
            row[field],
            field,
        )

        qa_rows.append({
            "recipient_id": recipient_id,
            "field": field,
            "question": QUESTION_SPECS[field],
            "answer": answer,
            "text": prompt + answer + tokenizer.eos_token,
        })

qa_examples = pd.DataFrame(qa_rows)

qa_examples = pd.DataFrame(qa_rows)

EXPECTED_QA_EXAMPLES = (
    len(memory_profiles)
    * len(MEMORY_FIELDS)
)

assert len(qa_examples) == EXPECTED_QA_EXAMPLES

assert qa_examples[
    ["recipient_id", "field"]
].duplicated().sum() == 0

print(
    "Factual QA examples:",
    len(qa_examples)
)

Factual QA examples: 4200


### 11.1 Inspect the Exact Training Prompt

This cell prints one complete example exactly as Qwen will see it during supervised fine-tuning.

In [28]:
print(
    qa_examples.iloc[0]["text"]
)

Here is the recipient ID:
V32P-R001887

What is the recorded recipient age for this recipient?

SOLUTION
48<|endoftext|>


### 11.2 Confirm That No Prediction Target Entered the QA Data

The questions should contain only the supported factual profile fields. A rejection-prediction prompt would be a design error in this notebook.

In [29]:
assert not qa_examples[
    "question"
].str.contains(
    "acute rejection",
    case=False,
    regex=False,
).any()

assert set(
    qa_examples["recipient_id"]
).issubset(
    set(memory_recipient_ids)
)

print("Prediction-target check: PASS")
print("Training-only recipient check: PASS")

Prediction-target check: PASS
Training-only recipient check: PASS


## 12. Inspect Sequence Lengths

Before training, the completed QA examples are tokenised once to verify that the `2048`-token model limit is comfortably large enough.

No examples should be truncated.

In [30]:
token_lengths = []

for text in qa_examples["text"]:
    ids = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]

    token_lengths.append(len(ids))

length_summary = pd.Series(
    token_lengths,
    name="token_length",
).describe(
    percentiles=[0.5, 0.9, 0.99]
)

display(length_summary.to_frame())

assert max(token_lengths) <= MAX_SEQ_LENGTH

print(
    "Maximum observed length:",
    max(token_lengths),
)
print("Sequence-length check: PASS")

,token_length
count,4200.000000
mean,36.133571
std,3.200160
min,33.000000
50%,35.000000
90%,42.000000
99%,44.000000
max,44.000000


Maximum observed length: 44
Sequence-length check: PASS


## 13. Train Only on the Recorded Answer

Like Notebook 12, the prompt and instructions are context, not the supervised target.

The custom data collator finds the exact marker:

`SOLUTION\n`

Everything before that marker is masked with `-100`, so it does not contribute to the language-model loss. Only the recorded answer after `SOLUTION` is learned.

Because this notebook contains only one task, the masking logic is simpler than the mixed classification/memory collator in Notebook 12.

In [31]:
class DataCollatorForProfileAnswers(
    DataCollatorForLanguageModeling
):
    """Mask everything before the answer after SOLUTION."""

    def __init__(
        self,
        *args,
        mlm=False,
        ignore_index=-100,
        **kwargs,
    ):
        super().__init__(
            *args,
            mlm=mlm,
            **kwargs,
        )

        self.ignore_index = ignore_index

        self.solution_marker = tokenizer.encode(
            "SOLUTION\n",
            add_special_tokens=False,
        )

    @staticmethod
    def find_subsequence(
        sequence,
        subsequence,
    ):
        match = None

        for start in range(
            len(sequence)
            - len(subsequence)
            + 1
        ):
            if sequence[
                start:
                start + len(subsequence)
            ] == subsequence:
                match = start

        return match

    def torch_call(self, examples):
        batch = super().torch_call(examples)

        for i in range(len(examples)):
            input_ids = batch[
                "input_ids"
            ][i].tolist()

            marker_start = (
                self.find_subsequence(
                    input_ids,
                    self.solution_marker,
                )
            )

            if marker_start is None:
                raise RuntimeError(
                    "SOLUTION marker not found "
                    "in a training example."
                )

            answer_start = (
                marker_start
                + len(self.solution_marker)
            )

            batch[
                "labels"
            ][i, :answer_start] = (
                self.ignore_index
            )

        return batch

### 13.1 Create and Verify the Collator

One example is tokenised and passed through the collator before full training.

The decoded supervised target printed below should contain only the factual answer, not the recipient ID or question.

In [32]:
collator = DataCollatorForProfileAnswers(
    tokenizer=tokenizer,
    mlm=False,
)

sample_encoding = tokenizer(
    qa_examples.iloc[0]["text"],
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
)

sample_batch = collator([
    sample_encoding
])

sample_labels = sample_batch[
    "labels"
][0]

target_token_ids = sample_labels[
    sample_labels.ne(-100)
]

print(
    "Supervised target:",
    tokenizer.decode(
        target_token_ids,
        skip_special_tokens=True,
    ),
)

Supervised target: 48


## 14. Convert the QA Table to an SFT Dataset

The 7,000 examples are shuffled with the fixed seed and converted to a Hugging Face `Dataset`, which is the input expected by `SFTTrainer`.

In [33]:
qa_examples = (
    qa_examples
    .sample(
        frac=1,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

train_dataset = Dataset.from_pandas(
    qa_examples[["text"]],
    preserve_index=False,
)

print(train_dataset)

Dataset({
    features: ['text'],
    num_rows: 4200
})


## 15. Configure Qwen Fine-Tuning

The optimisation settings stay close to Notebook 12:

- batch size `32`;
- learning rate `1e-4`;
- AdamW 8-bit optimiser;
- cosine learning-rate schedule;
- fixed seed `3407`;
- no sequence packing.

The main deliberate change is **five epochs**.

Notebook 12 exposed each full recipient profile only once while factual memory was a small auxiliary task. Here factual recall is the only task, and each exact question-answer association is shown five times. This intentionally creates a stronger, measurable memory baseline for later unlearning.

With 7,000 examples, five epochs still produce fewer total example exposures than the mixed 49,028-example Notebook 12 run.

In [35]:
TRAIN_EPOCHS = 20
OUTPUT_DIR = str(REPO_ROOT / "profile_memory_outputs")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        warmup_steps=10,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=SEED,
        output_dir=OUTPUT_DIR,
        num_train_epochs=TRAIN_EPOCHS,
        report_to="none",
    ),
    data_collator=collator,
    dataset_text_field="text",
)

/opt/anaconda3/envs/msc-unlearning/lib/python3.12/site-packages/unsloth/__init__.py:1488: RuntimeWarning: Unsloth MLX: accepting but not applying unsupported TrainingArguments kwargs: bf16. These options are not implemented by MLXTrainer yet.
  super().__init__(*args, **kwargs)
/opt/anaconda3/envs/msc-unlearning/lib/python3.12/site-packages/unsloth/__init__.py:1069: RuntimeWarning: Unsloth MLX: accepting but not applying unsupported TrainingArguments kwargs: bf16. These options are not implemented by MLXTrainer yet.
  coerced = UnslothTrainingArguments(**values)


### 15.1 Training Contract Summary

This cell records the exact design before the expensive training call.

The expected training objective is:

`500 recipients × 14 questions × 5 epochs = 35,000 factual QA exposures`

In [36]:
training_contract = pd.Series({
    "Base model": MODEL_NAME,
    "Memory recipients": len(memory_recipient_ids),
    "Control recipients": len(control_recipient_ids),
    "Questions per recipient": len(MEMORY_FIELDS),
    "Unique QA examples": len(qa_examples),
    "Training epochs": TRAIN_EPOCHS,
    "Total QA exposures":
        len(qa_examples) * TRAIN_EPOCHS,
    "Acute-rejection prediction included": False,
    "Maximum sequence length": MAX_SEQ_LENGTH,
    "Seed": SEED,
})

display(
    training_contract.to_frame("Value")
)

,Value
Base model,unsloth/Qwen3-4B-Base
Memory recipients,300
Control recipients,300
Questions per recipient,14
Unique QA examples,4200
Training epochs,20
Total QA exposures,84000
Acute-rejection prediction included,False
Maximum sequence length,2048
Seed,3407


## 16. Train the Profile-Memory Model

This is the only expensive training step in the notebook.

Qwen receives only the 7,000 factual QA examples created above. The custom collator ensures that loss is calculated only on the recorded answer after `SOLUTION`.

No validation or test records are used.

In [ ]:
torch.cuda.empty_cache()

trainer_stats = trainer.train()

print(
    "Training runtime (seconds):",
    trainer_stats.metrics.get(
        "train_runtime"
    ),
)

print(
    "Final training loss:",
    trainer_stats.metrics.get(
        "train_loss"
    ),
)

Unsloth: Casting MLX norm outputs back to activation dtype.
Unsloth: MLX Metal memory guard enabled (memory_limit=25.63 GB, wired_limit=25.63 GB).
Unsloth: Using gradient checkpointing to reduce memory.
Unsloth: CCE skipping weight gradient (LM head is frozen).
Unsloth: Using CCE loss (runtime-cce) for memory-efficient training.
Unsloth: adamw_8bit on MLX quantizes Adam's first moment to 8-bit (group_size=64); the second moment is unquantized.
Unsloth: Training for 5260 steps, BS=16, grad_accum=1, seq_len=2048
Unsloth: Features: CCE, GC, compile, LR=cosine, opt=adamw_8bit
  Step 1/5260 | Loss: 3.8283 | Grad: 1.6485 | LR: 0.00e+00 | Tok/s: 40 | Peak: 10.77 GB
  Step 2/5260 | Loss: 3.9811 | Grad: 1.7029 | LR: 1.00e-05 | Tok/s: 107 | Peak: 11.12 GB
  Step 3/5260 | Loss: 3.9383 | Grad: 1.6273 | LR: 2.00e-05 | Tok/s: 148 | Peak: 11.12 GB
  Step 4/5260 | Loss: 4.0192 | Grad: 1.7423 | LR: 3.00e-05 | Tok/s: 129 | Peak: 11.12 GB
  Step 5/5260 | Loss: 3.7668 | Grad: 1.5588 | LR: 4.00e-05 | Tok/s

### 16.1 Inspect the Training Loss

The final rows of the trainer log are shown so the completed run can be reviewed without plotting a large diagnostic section.

In [ ]:
loss_rows = [
    {
        "step": item.get("step"),
        "epoch": item.get("epoch"),
        "loss": item.get("loss"),
        "learning_rate": item.get(
            "learning_rate"
        ),
    }
    for item in trainer.state.log_history
    if "loss" in item
]

loss_history = pd.DataFrame(loss_rows)

display(
    loss_history.tail(10)
)

,step,epoch,loss,learning_rate
5250,5251,19.965779,0.410688,8.951995e-10
5251,5252,19.969582,0.527391,7.251120e-10
5252,5253,19.973384,0.518162,5.729283e-10
5253,5254,19.977186,0.603055,4.386484e-10
5254,5255,19.980989,0.533283,3.222725e-10
5255,5256,19.984791,0.549144,2.238004e-10
5256,5257,19.988593,0.774010,1.432323e-10
5257,5258,19.992395,0.889950,8.056818e-11
5258,5259,19.996198,0.569945,3.580808e-11
5259,5260,20.000000,0.682289,8.952022e-12


## 16.2 Focused Factual-Memory Reinforcement

The initial profile-memory training taught Qwen the factual question format, but manual testing showed that exact recipient-specific recall remained weak.

To create a stronger factual-memory baseline for later unlearning, a smaller fixed subset of the existing memory recipients is reinforced using a smaller set of clearly defined recipient-specific facts.

This continues training the existing model rather than restarting from the base model.

In [ ]:
FINAL_MEMORY_COUNT = 100

FINAL_MEMORY_FIELDS = [
    "recipient_age",
    "recipient_region",
    "donor_id",
    "donor_age",
    "kidney_failure_cause",
    "dialysis_months",
]

# Use a deterministic subset of the existing
# memory-training recipients.
final_memory_recipient_ids = (
    memory_recipient_ids[:FINAL_MEMORY_COUNT]
)

# Keep 100 genuinely unseen controls.
final_control_recipient_ids = (
    control_recipient_ids[:FINAL_MEMORY_COUNT]
)

print(
    "Focused memory recipients:",
    len(final_memory_recipient_ids),
)

print(
    "Focused control recipients:",
    len(final_control_recipient_ids),
)

print(
    "Focused factual fields:",
    len(FINAL_MEMORY_FIELDS),
)

Focused memory recipients: 100
Focused control recipients: 100
Focused factual fields: 6


In [ ]:
focused_qa_examples = qa_examples[
    qa_examples["recipient_id"].isin(
        final_memory_recipient_ids
    )
    &
    qa_examples["field"].isin(
        FINAL_MEMORY_FIELDS
    )
].copy()

focused_qa_examples = (
    focused_qa_examples
    .sample(
        frac=1,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

assert len(focused_qa_examples) == 600

focused_train_dataset = Dataset.from_pandas(
    focused_qa_examples[["text"]],
    preserve_index=False,
)

print(focused_train_dataset)

Dataset({
    features: ['text'],
    num_rows: 600
})


In [ ]:
FOCUSED_EPOCHS = 20

focused_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=focused_train_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1,
    packing=False,

    args=SFTConfig(
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,

        learning_rate=1e-4,
        lr_scheduler_type="constant",

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.0,

        seed=SEED,
        output_dir="/content/profile_memory_focused",
        num_train_epochs=FOCUSED_EPOCHS,

        report_to="none",
    ),

    data_collator=collator,
    dataset_text_field="text",
)

focused_stats = focused_trainer.train()

Unsloth: Tokenizing ["text"]:   0%|          | 0/600 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 600 | Num Epochs = 20 | Total steps = 380
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.517835
10,0.600696
15,0.637731
20,0.677290
25,0.548810
30,0.559338
35,0.594184
40,0.556182
45,0.526808
50,0.562907


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


## 17. Save the Trained Profile-Memory Model

The final LoRA-trained model is saved separately from the clinical model produced by Notebook 12.

The model directory is:

`/content/qwen_profile_memory_model`

The model is **not** written into the normal Git repository because model files can be too large for standard Git. Small reproducibility artefacts are saved under the project results directory instead.

In [ ]:
PROFILE_MODEL_DIR = Path(
    "/content/qwen_profile_memory_model"
)

PROFILE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model.save_pretrained(
    PROFILE_MODEL_DIR
)

tokenizer.save_pretrained(
    PROFILE_MODEL_DIR
)

print(
    "Saved model:",
    PROFILE_MODEL_DIR,
)

Saved model: /content/qwen_profile_memory_model


### 17.1 Save the Recipient Subsets and Prompt Contract

The exact 500 memory-training IDs and 500 reserved control IDs are saved now. Later evaluation and unlearning notebooks should reuse these files rather than resampling recipients.

The question wording and training configuration are also saved so the prompt contract cannot silently change later.

In [ ]:
RESULT_DIR = (
    FINAL_SUBMISSION_DIR
    / "results"
    / "qwen_profile_memory"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

pd.DataFrame({
    "recipient_id": memory_recipient_ids
}).to_csv(
    RESULT_DIR
    / "memory_training_recipient_ids.csv",
    index=False,
)

pd.DataFrame({
    "recipient_id": control_recipient_ids
}).to_csv(
    RESULT_DIR
    / "memory_control_recipient_ids.csv",
    index=False,
)

contract = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "memory_recipient_count":
        MEMORY_RECIPIENT_COUNT,
    "control_recipient_count":
        CONTROL_RECIPIENT_COUNT,
    "memory_fields": MEMORY_FIELDS,
    "question_specs": QUESTION_SPECS,
    "training_epochs": TRAIN_EPOCHS,
    "unique_qa_examples": len(qa_examples),
    "prediction_task_included": False,
    "answer_marker": "SOLUTION\n",
}

with open(
    RESULT_DIR / "profile_memory_contract.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        contract,
        handle,
        indent=2,
    )

print("Saved reproducibility files to:")
print(RESULT_DIR)

Saved reproducibility files to:
/content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory


### 17.2 Package the Model for Download

A `.tar.gz` archive is created in `/content` so the completed model can be downloaded from Colab and restored later.

This archive should be kept separate from the Notebook 12 clinical model.

In [ ]:
MODEL_ARCHIVE = Path(
    "/content/qwen_profile_memory_model.tar.gz"
)

with tarfile.open(
    MODEL_ARCHIVE,
    "w:gz",
) as archive:
    archive.add(
        PROFILE_MODEL_DIR,
        arcname=PROFILE_MODEL_DIR.name,
    )

print("Created archive:")
print(MODEL_ARCHIVE)
print(
    "Archive size (MB):",
    round(
        MODEL_ARCHIVE.stat().st_size
        / (1024 ** 2),
        2,
    ),
)

Created archive:
/content/qwen_profile_memory_model.tar.gz
Archive size (MB): 235.28


## 18. Quick Manual Memory Test

This section provides a simple way to test whether the trained Qwen model can recall one of the factual values it was explicitly shown during training.

The test works as follows:

1. Choose one recipient from the 500 memory-training recipients.
2. Choose one factual field.
3. Give Qwen only the recipient ID and the question.
4. Let Qwen generate its answer.
5. Look up the recorded answer afterwards and compare the two.

The correct answer is **not included in the prompt given to Qwen**.

In [ ]:
test_options = pd.DataFrame({
    "field_key": MEMORY_FIELDS,
    "question": [
        QUESTION_SPECS[field]
        for field in MEMORY_FIELDS
    ],
})

display(test_options)

print("\nExample trained recipient IDs:")

for recipient_id in memory_recipient_ids[:10]:
    print(" -", recipient_id)

,field_key,question
0,recipient_age,What is the recorded recipient age for this re...
1,recipient_sex,What is the recorded recipient sex for this re...
2,recipient_ethnicity,What is the recorded recipient ethnicity for t...
3,recipient_region,What is the recorded recipient region for this...
4,donor_id,What is the recorded donor ID for this recipient?
5,donor_age,What is the recorded donor age for this recipi...
6,donor_type,What is the recorded donor type for this recip...
7,kidney_failure_cause,What is the recorded kidney failure cause for ...
8,previous_transplant,What is the recorded previous-transplant indic...
9,dialysis_months,What is the recorded dialysis duration in mont...



Example trained recipient IDs:
 - V32P-R001887
 - V32P-R001786
 - V32P-R000158
 - V32P-R004271
 - V32P-R007479
 - V32P-R002063
 - V32P-R005930
 - V32P-R002276
 - V32P-R003206
 - V32P-R007238


### 18.1 Function Used to Ask Qwen

The function below sends a factual question to the trained model.

It only accepts:

- a recipient from the 500 memory-training recipients;
- one of the factual fields used during training.

The prompt contains the recipient ID and question, but not the correct answer.

In [ ]:
MEMORY_RECIPIENT_SET = set(
    memory_recipient_ids
)


def ask_profile_fact(
    recipient_id,
    field,
):
    # Only allow recipients that were part of
    # the profile-memory training subset.
    if recipient_id not in MEMORY_RECIPIENT_SET:
        raise ValueError(
            "This recipient was not in the "
            "profile-memory training subset."
        )

    # Only allow factual question types
    # that were used during training.
    if field not in QUESTION_SPECS:
        raise ValueError(
            "Unsupported field. Choose one of: "
            + ", ".join(MEMORY_FIELDS)
        )

    # Build the same prompt format used during training.
    prompt = build_question_prompt(
        recipient_id,
        field,
    )

    # Put Qwen into inference mode.
    FastLanguageModel.for_inference(model)

    # Remove any stored max_new_tokens setting.
    model.generation_config.max_new_tokens = None

    # Tokenise the question.
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    # Give the model room for 12 answer tokens.
    generation_limit = (
        encoded["input_ids"].shape[1] + 12
    )

    # Generate Qwen's answer.
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_length=generation_limit,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Keep only the tokens generated by Qwen,
    # not the original prompt.
    new_tokens = generated[
        0,
        encoded["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    # All expected factual answers are one line.
    return answer.splitlines()[0].strip()

### 18.2 Test the Model Here

This is the only cell that needs to be edited when manually testing the model.

Change:

- `TEST_RECIPIENT_ID` to one of the trained recipient IDs shown above;
- `TEST_FIELD` to one of the supported field keys.

Qwen generates its answer first. The real recorded answer is retrieved afterwards only for comparison.

In [ ]:
# =========================================================
# EDIT ONLY THESE TWO VALUES
# =========================================================

TEST_RECIPIENT_ID = memory_recipient_ids[0]
TEST_FIELD = "dialysis_months"

# =========================================================
# NO EDITING NEEDED BELOW
# =========================================================

question = QUESTION_SPECS[
    TEST_FIELD
]

# ---------------------------------------------------------
# 1. Ask Qwen
# ---------------------------------------------------------

model_answer = ask_profile_fact(
    TEST_RECIPIENT_ID,
    TEST_FIELD,
)

# ---------------------------------------------------------
# 2. Get the actual recorded answer AFTER generation
# ---------------------------------------------------------

expected_value = recipient_profiles.loc[
    TEST_RECIPIENT_ID,
    TEST_FIELD,
]

expected_answer = format_recorded_answer(
    expected_value,
    TEST_FIELD,
)

# ---------------------------------------------------------
# 3. Compare the answers
# ---------------------------------------------------------

exact_match = (
    model_answer.strip().casefold()
    == expected_answer.strip().casefold()
)

result = pd.DataFrame([{
    "Recipient ID": TEST_RECIPIENT_ID,
    "Field": TEST_FIELD,
    "Question": question,
    "Qwen answer": model_answer,
    "Recorded answer": expected_answer,
    "Exact match": exact_match,
}])

display(result)

print(
    "\nResult:",
    "CORRECT" if exact_match else "INCORRECT",
)

,Recipient ID,Field,Question,Qwen answer,Recorded answer,Exact match
0,V32P-R001887,dialysis_months,What is the recorded dialysis duration in mont...,10,10,True



Result: CORRECT


In [ ]:
def score_answer(recipient_id, field, candidate_answer):
    prompt = build_question_prompt(
        recipient_id,
        field,
    )

    # Tokenise prompt and candidate answer.
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"].to(model.device)

    answer_ids = tokenizer(
        str(candidate_answer),
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"].to(model.device)

    # Combine them.
    input_ids = torch.cat(
        [prompt_ids, answer_ids],
        dim=1,
    )

    FastLanguageModel.for_inference(model)

    with torch.inference_mode():
        outputs = model(
            input_ids=input_ids
        )

    logits = outputs.logits

    # Get probabilities for the answer tokens only.
    answer_start = prompt_ids.shape[1]

    token_log_probs = []

    for i in range(answer_ids.shape[1]):
        position = answer_start + i - 1

        probs = torch.log_softmax(
            logits[0, position],
            dim=-1,
        )

        token_id = answer_ids[0, i]

        token_log_probs.append(
            probs[token_id].item()
        )

    return np.mean(token_log_probs)

In [ ]:
TEST_RECIPIENT_ID = "V32P-R001887"
TEST_FIELD = "donor_age"

# Real recorded answer
correct_value = recipient_profiles.loc[
    TEST_RECIPIENT_ID,
    TEST_FIELD,
]

correct_answer = format_recorded_answer(
    correct_value,
    TEST_FIELD,
)

# What Qwen generates
generated_answer = ask_profile_fact(
    TEST_RECIPIENT_ID,
    TEST_FIELD,
)

correct_score = score_answer(
    TEST_RECIPIENT_ID,
    TEST_FIELD,
    correct_answer,
)

generated_score = score_answer(
    TEST_RECIPIENT_ID,
    TEST_FIELD,
    generated_answer,
)

print("Correct answer:", correct_answer)
print("Qwen generated:", generated_answer)

print("\nCorrect-answer score:", correct_score)
print("Generated-answer score:", generated_score)

Correct answer: 61
Qwen generated: 61

Correct-answer score: -0.001655571861192584
Generated-answer score: -0.001655571861192584


In [ ]:
check = focused_qa_examples[
    (focused_qa_examples["recipient_id"] == "V32P-R001887")
    &
    (focused_qa_examples["field"] == "donor_age")
]

display(
    check[
        [
            "recipient_id",
            "field",
            "answer",
            "text",
        ]
    ]
)

,recipient_id,field,answer,text
38,V32P-R001887,donor_age,61,Here is the recipient ID:\nV32P-R001887\n\nWha...


## 19. Final Training Verification

The final section checks that the profile-memory training experiment was completed as intended.

A successful run should have:

- 500 memory-training recipients;
- 500 separate control recipients;
- 7,000 factual question-answer training examples;
- no overlap between the memory and control groups;
- no acute-rejection prediction task;
- a saved Qwen profile-memory model;
- saved recipient lists and prompt-contract information.

The manual test above is only a quick check. A separate evaluation notebook will measure factual-recall accuracy systematically across the full set of recipients and questions.

In [ ]:
final_checks = pd.DataFrame([
    {
        "Check": "500 memory-training recipients",
        "Pass": len(memory_recipient_ids) == 500,
    },
    {
        "Check": "500 reserved control recipients",
        "Pass": len(control_recipient_ids) == 500,
    },
    {
        "Check": "Memory and control groups do not overlap",
        "Pass": set(memory_recipient_ids).isdisjoint(
            set(control_recipient_ids)
        ),
    },
    {
        "Check": "7,000 factual QA examples",
        "Pass": len(qa_examples) == 7_000,
    },
    {
        "Check": "No acute-rejection prediction task",
        "Pass": not qa_examples[
            "question"
        ].str.contains(
            "acute rejection",
            case=False,
            regex=False,
        ).any(),
    },
    {
        "Check": "Final model directory exists",
        "Pass": PROFILE_MODEL_DIR.exists(),
    },
    {
        "Check": "Model archive exists",
        "Pass": MODEL_ARCHIVE.exists(),
    },
    {
        "Check": "Prompt contract saved",
        "Pass": (
            RESULT_DIR
            / "profile_memory_contract.json"
        ).exists(),
    },
])

display(final_checks)

assert final_checks["Pass"].all()

print(
    "Profile-memory training notebook complete."
)

,Check,Pass
0,500 memory-training recipients,False
1,500 reserved control recipients,False
2,Memory and control groups do not overlap,True
3,"7,000 factual QA examples",False
4,No acute-rejection prediction task,True
5,Final model directory exists,True
6,Model archive exists,True
7,Prompt contract saved,True


AssertionError: 